# Private local ORB-SLAM → Track A v10 builder

이 노트북은 선택한 로컬 경로만 읽고 저장소의 기존 Track A 도구를 호출합니다. 네트워크 업로드, 원격 API 호출, 학습 및 정책 추론을 수행하지 않습니다.

중요한 제한:

- HW 전달 데이터셋 builder와 patched ORB-SLAM3 실행 파일의 **로컬 경로**를 지정하면 telemetry·gripper 산출, mapping Atlas, episode localization trajectory, ArUco 정렬부터 v10까지 실행합니다. ORB-SLAM3 자체는 C++ 네이티브 프로세스이며 Jupyter는 인수를 검증한 뒤 `subprocess` 배열로 호출합니다.
- 저장소에는 ORB-SLAM3 실행 파일과 vocabulary가 포함되어 있지 않습니다. HW가 사용한 패치 적용 빌드를 지정해야 하며, 미패치/다른 빌드로 생성한 Atlas를 같은 후보로 혼합하면 안 됩니다.
- 고정 마커 계약은 `DICT_4X4_50`, ID 13, 검정 사각형 0.16 m입니다. 다른 수집 리그는 설정과 검증 근거를 새로 만들어야 합니다.
- 기본 camera→TCP는 현재 축 정정된 CAD provisional 설정입니다. 실물 배포 승인을 뜻하지 않습니다.
- 생성되는 v10 NPZ에는 224×224 RGB 프레임이 포함됩니다. **원본만 로컬에 두고 v10을 공유해도 집 내부 영상은 v10 안에 남습니다.** 프라이버시가 필요하면 v10/TAR도 외부로 보내지 말고 동일 환경에서 학습한 체크포인트와 비식별 검증 요약만 전달하십시오.
- 새 데이터에 과거 에피소드의 gap 예외를 자동 적용하지 않습니다. 새 예외는 프레임을 직접 검토하고 별도 JSON으로 명시해야 합니다.


In [ ]:
from pathlib import Path
import hashlib
import json
import os
import subprocess
import sys

def find_repo(start: Path) -> Path:
    for candidate in [start.resolve(), *start.resolve().parents]:
        if (candidate / 'AI' / 'tools').is_dir() and (candidate / '.git').exists():
            return candidate
    raise RuntimeError('저장소 루트에서 노트북을 실행하거나 REPO_ROOT를 직접 지정하세요.')

REPO_ROOT = find_repo(Path.cwd())
AI_ROOT = REPO_ROOT / 'AI'
TOOLS = AI_ROOT / 'tools'
print('repo:', REPO_ROOT)
print('python:', sys.executable)


## 1. raw→ORB→v10 경로 설정

아래 경로만 자신의 Jupyter 환경에 맞게 수정합니다. `BUILDER_ROOT`는 `handoff/final/02_umi_dataset_builder`, `ORB_SLAM_BIN`은 Track A 패치가 적용된 `gopro_slam`입니다.

네이티브 ORB 실행과 v10 생성은 각각 별도 확인 플래그를 사용합니다. 첫 실행에서는 둘 다 `False`로 두고 경로·계약·selection을 검토하십시오.


In [ ]:
# 원본: MAPPING_SESSION 1편 + RAW_ROOT/rec_* 시연들
MAPPING_SESSION = Path('/home/USER/private_raw/raw_mapping')
RAW_ROOT = Path('/home/USER/private_raw/episodes')

# HW 전달 패키지를 로컬에 풀어놓은 경로
BUILDER_ROOT = Path('/home/USER/handoff/final/02_umi_dataset_builder')
BUILDER = BUILDER_ROOT / 'build_dataset.py'
ORB_SETTINGS = BUILDER_ROOT / 'slam/s22_mono_inertial.yaml'
ORB_MASK = BUILDER_ROOT / 'configs/s22_slam_mask.png'

# Track A orbslam3_umi_intake_0918.patch가 적용된 native build
ORB_SLAM_BIN = Path('/home/USER/third_party/ORB_SLAM3/Examples/Monocular-Inertial/gopro_slam')
ORB_SLAM_VOCAB = Path('/home/USER/third_party/ORB_SLAM3/Vocabulary/ORBvoc.txt')

# 중간 산출물은 원본과 분리합니다. 이미 완료된 episode는 재개 시 건너뜁니다.
WORK_ROOT = Path('/home/USER/private_orbslam_work')
PREPARED_ROOT = WORK_ROOT / 'prepared'
MAPPING_PREPARED = WORK_ROOT / 'mapping_prepared'
MAPPING_ORB_ROOT = WORK_ROOT / 'mapping_orb'
ATLAS_PATH = WORK_ROOT / 'atlas/map1.osa'
ALIGNMENT_JSON = WORK_ROOT / 'atlas/map1_alignment.json'
BATCH_ROOT = WORK_ROOT / 'map1_episode_runs'
GRIPPER_ROOT = PREPARED_ROOT
CANDIDATES = [{'name': 'map1', 'atlas': ATLAS_PATH, 'alignment': ALIGNMENT_JSON, 'batch_root': BATCH_ROOT}]

# 축 정정된 최신 Track A CAD provisional 설정. 다른 장착이면 교체해야 합니다.
CAMERA_TCP = AI_ROOT / 'configs/real/s22_camera_tcp_cad_video_aligned_provisional_0918.json'

# 출력은 원본과 분리된 새 디렉터리로 지정합니다. 기존 경로는 덮어쓰지 않습니다.
OUTPUT_ROOT = WORK_ROOT / 'v10_output'
DATASET_NAME = 'umi_real_relative_PRIVATE_v10_orbslam_provisional'

# 품질 기준: 0918 Track A reference와 동일. 결과를 본 뒤 완화하지 않습니다.
MINIMUM_COVERAGE = 0.95
MAXIMUM_FIRST_TIME_S = 0.5
MAXIMUM_TIMESTAMP_ERROR_MS = 2.0
MAXIMUM_GRIPPER_MISSING_RUN = 15

# 새 데이터에는 과거 0918 예외를 쓰지 않습니다. 직접 검토한 새 JSON만 지정하세요.
REVIEWED_GAP_EXCEPTIONS = None  # 예: Path(r'C:\...\reviewed_gap_exceptions.json')

# 동일한 실측 ID 13 보드일 때만 True. 다른 인쇄물/리그면 False로 유지합니다.
FIXED_MARKER_SIZE_VERIFIED = True
# 빈 그리퍼 전행정 등으로 이번 장착의 metric gap scale을 확인했을 때만 True.
GAP_SCALE_VERIFIED = False

# 안전 확인: True로 바꾸기 전 아래 preflight 출력을 검토하십시오.
CONFIRM_NATIVE_ORB_RUN = False

SELECTION_JSON = OUTPUT_ROOT / 'atlas_selection.json'
SELECTION_GATE_JSON = OUTPUT_ROOT / 'episode_selection_gate_report.json'
SELECTION_GATE_CSV = OUTPUT_ROOT / 'episode_selection_gate_report.csv'
V10_ROOT = OUTPUT_ROOT / DATASET_NAME
VERIFY_JSON = OUTPUT_ROOT / 'verification_report.json'
FINAL_GATE_JSON = OUTPUT_ROOT / 'episode_gate_report.json'
FINAL_GATE_CSV = OUTPUT_ROOT / 'episode_gate_report.csv'
REFERENCE_ROOT = OUTPUT_ROOT / f'{DATASET_NAME}_reference'
REFERENCE_TAR = OUTPUT_ROOT / f'{DATASET_NAME}_reference.tar'


In [ ]:
# 입력과 계약을 읽기 전용으로 점검합니다.
def require_file(path: Path, label: str):
    if not path.is_file():
        raise FileNotFoundError(f'{label}: {path}')

def require_dir(path: Path, label: str):
    if not path.is_dir():
        raise FileNotFoundError(f'{label}: {path}')

for value in [MAPPING_SESSION, RAW_ROOT, BUILDER_ROOT, ORB_SLAM_BIN, ORB_SLAM_VOCAB, WORK_ROOT]:
    if 'CHANGE_ME' in str(value):
        raise RuntimeError('먼저 CHANGE_ME 경로를 실제 로컬 경로로 바꾸세요.')
require_dir(MAPPING_SESSION, 'MAPPING_SESSION')
require_dir(RAW_ROOT, 'RAW_ROOT')
require_dir(BUILDER_ROOT, 'BUILDER_ROOT')
require_file(BUILDER, 'BUILDER')
require_file(ORB_SETTINGS, 'ORB_SETTINGS')
require_file(ORB_MASK, 'ORB_MASK')
require_file(ORB_SLAM_BIN, 'ORB_SLAM_BIN')
require_file(ORB_SLAM_VOCAB, 'ORB_SLAM_VOCAB')
if os.name != 'nt' and not os.access(ORB_SLAM_BIN, os.X_OK):
    raise PermissionError(f'ORB_SLAM_BIN이 실행 가능하지 않습니다: {ORB_SLAM_BIN}')
help_probe = subprocess.run([str(ORB_SLAM_BIN), '--help'], cwd=ORB_SLAM_BIN.parent, text=True, capture_output=True)
help_text = (help_probe.stdout or '') + '\n' + (help_probe.stderr or '')
required_orb_flags = ['--use_sensor_timestamps', '--localization_only', '--save_map', '--load_map']
missing_flags = [flag for flag in required_orb_flags if flag not in help_text]
if help_probe.returncode != 0 or missing_flags:
    raise RuntimeError(f'ORB-SLAM3 빌드/패치 preflight 실패: exit={help_probe.returncode}, missing={missing_flags}')
require_file(CAMERA_TCP, 'CAMERA_TCP')

camera_cfg = json.loads(CAMERA_TCP.read_text(encoding='utf-8'))
if 'T_camera_tcp' not in camera_cfg:
    raise ValueError('camera config에 T_camera_tcp가 없습니다.')

raw_episodes = sorted(p for p in RAW_ROOT.glob('rec_*') if p.is_dir())
if not raw_episodes:
    raise RuntimeError('RAW_ROOT 아래 rec_* episode가 없습니다.')

for output in [V10_ROOT, VERIFY_JSON, REFERENCE_ROOT, REFERENCE_TAR]:
    if output.exists():
        raise FileExistsError(f'기존 출력을 덮어쓰지 않습니다: {output}')

print('raw episodes:', len(raw_episodes))
for session in [MAPPING_SESSION, *raw_episodes]:
    for name in ['manifest.json', 'video.mp4', 'encoded.csv', 'frames.csv', 'accelerometer.csv', 'gyroscope.csv']:
        require_file(session / name, f'{session.name}/{name}')

print('camera calibration:', camera_cfg.get('calibration_id'), camera_cfg.get('status'))
print('ORB binary:', ORB_SLAM_BIN)
print('mapping session:', MAPPING_SESSION)
print('gap metric verified:', GAP_SCALE_VERIFIED)
print('privacy warning: v10 NPZ에도 RGB 프레임이 포함됩니다.')


In [ ]:
def run_tool(label: str, script: str, arguments: list[str]) -> None:
    command = [sys.executable, str(TOOLS / script), *map(str, arguments)]
    print(f'\n[{label}]')
    print(' '.join(command))
    result = subprocess.run(command, cwd=REPO_ROOT, text=True)
    if result.returncode != 0:
        raise RuntimeError(f'{label} 실패(exit={result.returncode}). 위 로그를 확인하세요.')

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)


## 2. telemetry·gripper 전처리

HW builder의 `prepare`를 mapping 녹화와 모든 `rec_*`에 적용합니다. `telemetry.json`, `gripper_width.csv`, `gripper_report.json`, `prepare_report.json`이 모두 있는 세션은 건너뜁니다. 일부 파일만 남은 디렉터리는 자동으로 덮어쓰지 않습니다.


In [ ]:
def run_command(label: str, command: list[object], *, cwd: Path | None = None, log: Path | None = None) -> None:
    command = [str(value) for value in command]
    print(f'\n[{label}]')
    print(' '.join(command))
    if log is None:
        result = subprocess.run(command, cwd=cwd, text=True)
    else:
        log.parent.mkdir(parents=True, exist_ok=True)
        with log.open('x', encoding='utf-8') as stream:
            result = subprocess.run(command, cwd=cwd, text=True, stdout=stream, stderr=subprocess.STDOUT)
    if result.returncode != 0:
        raise RuntimeError(f'{label} 실패(exit={result.returncode}); log={log}')

def file_sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open('rb') as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

def prepare_session(session: Path, output: Path) -> str:
    required = [output / name for name in ('telemetry.json', 'gripper_width.csv', 'gripper_report.json', 'prepare_report.json', 's22_mono_inertial.yaml')]
    if all(path.is_file() for path in required):
        return 'skip-complete'
    if output.exists() and any(output.iterdir()):
        raise RuntimeError(f'부분 prepare 출력을 덮어쓰지 않습니다: {output}')
    output.parent.mkdir(parents=True, exist_ok=True)
    run_command(f'prepare {session.name}', [sys.executable, BUILDER, 'prepare', session, output], cwd=BUILDER_ROOT)
    return 'created'

prepare_results = {'mapping': prepare_session(MAPPING_SESSION, MAPPING_PREPARED)}
for episode in raw_episodes:
    prepare_results[episode.name] = prepare_session(episode, PREPARED_ROOT / episode.name)
print(json.dumps({
    'created': sum(value == 'created' for value in prepare_results.values()),
    'skipped_complete': sum(value == 'skip-complete' for value in prepare_results.values()),
    'total': len(prepare_results),
}, indent=2))


## 3. mapping Atlas 생성

이 셀부터는 native ORB-SLAM3를 실행합니다. `CONFIRM_NATIVE_ORB_RUN=True`로 변경하기 전에 binary·vocabulary·S22 설정·ID 13/0.16m을 확인하십시오. Atlas와 mapping trajectory가 모두 있으면 재개 시 건너뜁니다.


In [ ]:
def orb_command(session: Path, prepared: Path, trajectory: Path, *, save_atlas: Path | None = None, load_atlas: Path | None = None) -> list[object]:
    command: list[object] = [
        ORB_SLAM_BIN, '--vocabulary', ORB_SLAM_VOCAB, '--setting', prepared / 's22_mono_inertial.yaml',
        '--input_video', session / 'video.mp4', '--input_imu_json', prepared / 'telemetry.json',
        '--output_trajectory_csv', trajectory, '--output_trajectory_tum', trajectory.with_name('camera_trajectory_tum.txt'),
        '--use_sensor_timestamps', '--init_tag_id', '13', '--init_tag_size', '0.16', '--mask_img', ORB_MASK,
    ]
    if save_atlas is not None:
        command += ['--save_map', save_atlas]
    if load_atlas is not None:
        command += ['--load_map', load_atlas, '--localization_only']
    return command

mapping_trajectory = MAPPING_ORB_ROOT / 'camera_trajectory.csv'
mapping_tracking = Path(str(mapping_trajectory) + '.tracking.csv')
mapping_complete = ATLAS_PATH.is_file() and mapping_trajectory.is_file() and mapping_tracking.is_file()
print('mapping complete:', mapping_complete)
print('atlas:', ATLAS_PATH)
print('command preview:', ' '.join(map(str, orb_command(MAPPING_SESSION, MAPPING_PREPARED, mapping_trajectory, save_atlas=ATLAS_PATH))))
if not mapping_complete:
    if CONFIRM_NATIVE_ORB_RUN is not True:
        raise RuntimeError('경로와 command preview를 검토한 뒤 CONFIRM_NATIVE_ORB_RUN=True로 변경하세요.')
    if MAPPING_ORB_ROOT.exists() and any(MAPPING_ORB_ROOT.iterdir()):
        raise RuntimeError(f'부분 mapping 출력을 덮어쓰지 않습니다: {MAPPING_ORB_ROOT}')
    MAPPING_ORB_ROOT.mkdir(parents=True, exist_ok=True)
    ATLAS_PATH.parent.mkdir(parents=True, exist_ok=True)
    run_command('build mapping Atlas', orb_command(MAPPING_SESSION, MAPPING_PREPARED, mapping_trajectory, save_atlas=ATLAS_PATH), cwd=ORB_SLAM_BIN.parent, log=MAPPING_ORB_ROOT / 'orbslam.log')
    require_file(ATLAS_PATH, 'generated Atlas')
    require_file(mapping_trajectory, 'mapping trajectory')
    require_file(mapping_tracking, 'mapping tracking sidecar')


## 4. Atlas↔ArUco ID 13 정렬

mapping 기간의 ORB camera pose와 영상에서 검출한 ID 13을 결합해 공통 작업대 좌표를 만듭니다. 실측 0.16m 보드가 아니면 PASS로 승격하지 않습니다.


In [ ]:
if not ALIGNMENT_JSON.exists():
    align_args = [
        '--video', MAPPING_SESSION / 'video.mp4', '--trajectory', mapping_trajectory,
        '--marker-id', '13', '--marker-size-m', '0.16', '--out', ALIGNMENT_JSON,
    ]
    if FIXED_MARKER_SIZE_VERIFIED:
        align_args.append('--marker-size-verified')
    run_tool('align Atlas to ArUco ID 13', 'align_orbslam_to_aruco.py', align_args)
alignment = json.loads(ALIGNMENT_JSON.read_text(encoding='utf-8'))
print(json.dumps({key: alignment.get(key) for key in ('status', 'observations', 'inliers', 'residuals')}, ensure_ascii=False, indent=2))
if alignment.get('status') != 'PASS_ARUCO_ALIGNMENT':
    raise RuntimeError(f"ArUco 정렬이 PASS가 아닙니다: {alignment.get('status')}")
inputs = alignment.get('inputs', {})
if inputs.get('marker_id') != 13 or abs(float(inputs.get('marker_black_square_size_m', -1)) - 0.16) > 1e-9:
    raise RuntimeError('alignment의 ID 13 / 0.16m 계약이 일치하지 않습니다.')
if Path(inputs.get('video', '')).resolve() != (MAPPING_SESSION / 'video.mp4').resolve():
    raise RuntimeError('alignment video가 현재 mapping session과 다릅니다.')
if Path(inputs.get('trajectory', '')).resolve() != mapping_trajectory.resolve():
    raise RuntimeError('alignment trajectory가 현재 mapping run과 다릅니다.')
if inputs.get('marker_size_verified') is not True:
    raise RuntimeError('ID 13 0.16m 실측 검증이 없어 PASS 후보로 사용할 수 없습니다.')


## 5. episode localization-only trajectory

공통 Atlas를 불러와 각 episode를 `--localization_only`로 처리합니다. 출력은 `rec_*/attempt0/`에 보존하며, 완성된 trajectory는 건너뜁니다. 실패 후 다시 시도할 때는 기존 attempt를 지우지 말고 `ATTEMPT_NUMBER`를 늘립니다.


In [ ]:
ATTEMPT_NUMBER = 0
if CONFIRM_NATIVE_ORB_RUN is not True:
    raise RuntimeError('episode localization 전 CONFIRM_NATIVE_ORB_RUN=True를 명시하세요.')
require_file(ATLAS_PATH, 'Atlas')
localization_results = {}
for index, episode in enumerate(raw_episodes, start=1):
    attempt = BATCH_ROOT / episode.name / f'attempt{ATTEMPT_NUMBER}'
    trajectory = attempt / 'camera_trajectory.csv'
    tracking = Path(str(trajectory) + '.tracking.csv')
    if trajectory.is_file() and tracking.is_file():
        localization_results[episode.name] = 'skip-complete'
        continue
    if attempt.exists() and any(attempt.iterdir()):
        raise RuntimeError(f'부분 localization 출력: {attempt}; ATTEMPT_NUMBER를 늘려 재시도하세요.')
    attempt.mkdir(parents=True, exist_ok=True)
    print(f'[{index}/{len(raw_episodes)}] {episode.name}')
    run_command(f'localize {episode.name}', orb_command(episode, PREPARED_ROOT / episode.name, trajectory, load_atlas=ATLAS_PATH), cwd=ORB_SLAM_BIN.parent, log=attempt / 'orbslam.log')
    require_file(trajectory, f'{episode.name} trajectory')
    require_file(tracking, f'{episode.name} tracking sidecar')
    localization_results[episode.name] = 'created'
print(json.dumps({
    'created': sum(value == 'created' for value in localization_results.values()),
    'skipped_complete': sum(value == 'skip-complete' for value in localization_results.values()),
    'total': len(localization_results),
}, indent=2))

orb_manifest = {
    'schema': 'private_orbslam_run/0.1',
    'network_upload_performed': False,
    'fixed_marker': {'dictionary': 'DICT_4X4_50', 'id': 13, 'black_square_size_m': 0.16, 'verified': FIXED_MARKER_SIZE_VERIFIED},
    'artifacts': {
        'orb_binary': {'name': ORB_SLAM_BIN.name, 'sha256': file_sha256(ORB_SLAM_BIN)},
        'vocabulary': {'name': ORB_SLAM_VOCAB.name, 'sha256': file_sha256(ORB_SLAM_VOCAB)},
        'settings': {'name': ORB_SETTINGS.name, 'sha256': file_sha256(ORB_SETTINGS)},
        'mask': {'name': ORB_MASK.name, 'sha256': file_sha256(ORB_MASK)},
        'builder': {'name': BUILDER.name, 'sha256': file_sha256(BUILDER)},
        'track_a_patch': {'name': 'orbslam3_umi_intake_0918.patch', 'sha256': file_sha256(AI_ROOT / 'third_party/orbslam3_umi_intake_0918.patch')},
        'atlas': {'name': ATLAS_PATH.name, 'sha256': file_sha256(ATLAS_PATH)},
        'alignment': {'name': ALIGNMENT_JSON.name, 'sha256': file_sha256(ALIGNMENT_JSON)},
    },
    'episodes': [{'episode_id': name, 'attempt': ATTEMPT_NUMBER} for name in sorted(localization_results)],
}
orb_manifest_path = WORK_ROOT / 'orb_run_manifest.json'
if orb_manifest_path.exists():
    existing = json.loads(orb_manifest_path.read_text(encoding='utf-8'))
    if existing != orb_manifest:
        raise RuntimeError(f'기존 ORB manifest와 현재 입력이 다릅니다: {orb_manifest_path}')
else:
    orb_manifest_path.write_text(json.dumps(orb_manifest, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
print('ORB manifest:', orb_manifest_path)


## 6. Atlas run 선택

에피소드마다 하나의 완전한 ORB run만 선택합니다. 서로 다른 run의 pose 행을 섞지 않습니다. 새 데이터의 outcome, gap 품질, 추적률, 최초 pose 시각, timestamp 오차를 함께 검사합니다.


In [ ]:
selection_args = [
    '--raw-root', RAW_ROOT,
    '--gripper-root', GRIPPER_ROOT,
    '--minimum-coverage', str(MINIMUM_COVERAGE),
    '--maximum-first-time-s', str(MAXIMUM_FIRST_TIME_S),
    '--maximum-timestamp-error-ms', str(MAXIMUM_TIMESTAMP_ERROR_MS),
    '--maximum-gripper-missing-run', str(MAXIMUM_GRIPPER_MISSING_RUN),
]
for candidate in CANDIDATES:
    selection_args += [
        '--candidate', candidate['name'], candidate['atlas'],
        candidate['alignment'], candidate['batch_root'],
    ]
if REVIEWED_GAP_EXCEPTIONS is not None:
    require_file(REVIEWED_GAP_EXCEPTIONS, 'REVIEWED_GAP_EXCEPTIONS')
    selection_args += ['--reviewed-gap-exceptions', REVIEWED_GAP_EXCEPTIONS]
selection_args += ['--out', SELECTION_JSON]
if SELECTION_JSON.exists():
    print('skip existing selection:', SELECTION_JSON)
else:
    run_tool('select ORB Atlas runs', 'select_orbslam_atlas_runs.py', selection_args)


In [ ]:
selection = json.loads(SELECTION_JSON.read_text(encoding='utf-8'))
if SELECTION_GATE_JSON.exists() != SELECTION_GATE_CSV.exists():
    raise RuntimeError('편별 selection 보고서 JSON/CSV 중 하나만 있습니다.')
if not SELECTION_GATE_JSON.exists():
    run_tool('export per-episode selection gate report', 'export_orbslam_episode_gate_report.py', [
        '--raw-root', RAW_ROOT, '--gripper-root', GRIPPER_ROOT, '--selection', SELECTION_JSON,
        '--out-json', SELECTION_GATE_JSON, '--out-csv', SELECTION_GATE_CSV,
    ])
else:
    prior_gate = json.loads(SELECTION_GATE_JSON.read_text(encoding='utf-8'))
    if prior_gate.get('inputs', {}).get('selection_sha256') != file_sha256(SELECTION_JSON):
        raise RuntimeError('기존 편별 selection 보고서가 현재 selection과 다릅니다.')
accepted = selection.get('episodes', [])
excluded = selection.get('excluded', {})
print('accepted:', len(accepted))
print('excluded:', len(excluded))
print('thresholds:', json.dumps(selection.get('thresholds', {}), ensure_ascii=False, indent=2))
print('\n첫 제외 사유 최대 20개:')
for episode_id, reasons in list(excluded.items())[:20]:
    print('-', episode_id, ':', reasons)
if not accepted:
    raise RuntimeError('채택 episode가 없습니다. 기준을 사후 완화하지 말고 입력 산출물을 점검하세요.')
print('\n검토가 끝났으면 v10 생성 셀의 CONFIRM_SELECTION_FOR_BUILD만 True로 바꾸세요.')


## 7. v10 생성과 계약 검증

`CONFIRM_SELECTION_FOR_BUILD=True`일 때만 H=2, K=8, 10Hz, 상대 EEF SE(3)+절대 gap 형식으로 생성합니다. camera→TCP 또는 gap scale이 provisional이면 dataset 상태도 자동으로 provisional이 됩니다.


In [ ]:
# 바로 아래 한 줄은 위 selection 결과와 제외 사유를 직접 검토한 뒤에만 바꿉니다.
CONFIRM_SELECTION_FOR_BUILD = False
if CONFIRM_SELECTION_FOR_BUILD is not True:
    raise RuntimeError('selection을 검토한 뒤 이 셀의 CONFIRM_SELECTION_FOR_BUILD=True로 명시하세요.')

build_args = [
    '--selection', SELECTION_JSON,
    '--camera-tcp', CAMERA_TCP,
    '--out', V10_ROOT,
    '--rate-hz', '10',
    '--obs-horizon', '2',
    '--action-horizon', '8',
]
if GAP_SCALE_VERIFIED:
    build_args.append('--gap-scale-verified')
run_tool('build Track A v10', 'build_orbslam_umi_relative_dataset.py', build_args)
run_tool('verify Track A v10', 'verify_umi_relative_dataset.py', [V10_ROOT, '--out', VERIFY_JSON])

dataset_index = json.loads((V10_ROOT / 'dataset.json').read_text(encoding='utf-8'))
verification = json.loads(VERIFY_JSON.read_text(encoding='utf-8'))
dataset_index_path = V10_ROOT / 'dataset.json'
if FINAL_GATE_JSON.exists() != FINAL_GATE_CSV.exists():
    raise RuntimeError('최종 편별 보고서 JSON/CSV 중 하나만 있습니다.')
if not FINAL_GATE_JSON.exists():
    run_tool('export final per-episode gate report', 'export_orbslam_episode_gate_report.py', [
        '--raw-root', RAW_ROOT, '--gripper-root', GRIPPER_ROOT, '--selection', SELECTION_JSON,
        '--dataset-index', dataset_index_path, '--scan-fixed-marker',
        '--out-json', FINAL_GATE_JSON, '--out-csv', FINAL_GATE_CSV,
    ])
else:
    prior_gate = json.loads(FINAL_GATE_JSON.read_text(encoding='utf-8'))
    expected_inputs = prior_gate.get('inputs', {})
    if (expected_inputs.get('selection_sha256') != file_sha256(SELECTION_JSON)
            or expected_inputs.get('dataset_index_sha256') != file_sha256(dataset_index_path)):
        raise RuntimeError('기존 최종 편별 보고서가 현재 selection/dataset과 다릅니다.')
print(json.dumps({
    'status': dataset_index.get('status'),
    'schema': dataset_index.get('schema'),
    'episodes': dataset_index.get('n_episodes'),
    'anchors': dataset_index.get('n_rows'),
    'problems': verification.get('problems'),
}, ensure_ascii=False, indent=2))


## 8. Track A 기준 묶음 생성·재검증

에피소드별 `pose.csv`, `gripper.csv`, manifest, 검증 기준과 체크섬을 만듭니다. 이 reference TAR와 v10 모두 RGB/로컬 provenance를 포함할 수 있으므로 자동 업로드하지 않습니다.


In [ ]:
run_tool('export Track A reference bundle', 'export_orbslam_v10_reference_bundle.py', [
    '--selection', SELECTION_JSON,
    '--camera-tcp', CAMERA_TCP,
    '--v10', V10_ROOT,
    '--out', REFERENCE_ROOT,
    '--tar', REFERENCE_TAR,
    '--episode-gate-report-json', FINAL_GATE_JSON,
    '--episode-gate-report-csv', FINAL_GATE_CSV,
    '--redact-source-paths',
])
run_tool('verify Track A reference bundle', 'check_orbslam_v10_reference_bundle.py', [REFERENCE_ROOT])


In [ ]:
def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open('rb') as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

summary = {
    'dataset': str(V10_ROOT),
    'dataset_json_sha256': sha256_file(V10_ROOT / 'dataset.json'),
    'episodes': dataset_index.get('n_episodes'),
    'anchors': dataset_index.get('n_rows'),
    'schema': dataset_index.get('schema'),
    'status': dataset_index.get('status'),
    'reference_tar': str(REFERENCE_TAR),
    'reference_tar_sha256': sha256_file(REFERENCE_TAR),
    'episode_gate_report_json': str(FINAL_GATE_JSON),
    'episode_gate_report_csv': str(FINAL_GATE_CSV),
    'network_upload_performed': False,
    'privacy_note': 'v10 contains RGB frames; review before sharing',
}
summary_path = OUTPUT_ROOT / 'LOCAL_BUILD_SUMMARY.json'
summary_path.write_text(json.dumps(summary, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
print(json.dumps(summary, ensure_ascii=False, indent=2))
print('\n외부 전달 전 dataset/TAR 내부 RGB와 절대 경로 provenance를 별도로 검토하세요.')
